In [5]:
from pathlib import Path
import time
import torch
from tqdm.auto import tqdm


# Input/output paths
INPUT_PATH = Path("../logs/rnn/2026-04-03_17-52-31_037802_rnn_pid114858/checkpoints/validation_only_ctc_export.pt").resolve()
OUTPUT_PATH = INPUT_PATH.with_name("validation_only_ctc_analysis_ready.pt")

BOUNDARY_RADIUS = 2
BLANK_ID = 0

VOWELS = {
    "AA", "AE", "AH", "AO", "AW", "AY",
    "EH", "ER", "EY", "IH", "IY", "OW",
    "OY", "UH", "UW",
}


def load_export(path: Path):
    print(f"Loading validation export from: {path}")
    t0 = time.time()
    payload = torch.load(path, map_location="cpu", weights_only=False)
    required = {
        "logits",
        "probabilities",
        "argmax_predictions",
        "decoded_ctc_sequences",
        "target_sequences",
        "input_lengths",
        "target_lengths",
        "phoneme_vocab",
    }
    missing = required.difference(payload.keys())
    if missing:
        raise KeyError(f"Missing keys in saved export: {sorted(missing)}")
    print(f"Loaded export in {time.time() - t0:.2f}s")
    return payload


def build_group_lookup(phoneme_vocab):
    group_lookup = {}
    for idx, token in enumerate(phoneme_vocab):
        token = str(token).strip()
        if idx == BLANK_ID or token == "BLANK":
            group_lookup[idx] = 0  # blank
        elif token in VOWELS:
            group_lookup[idx] = 1  # vowel
        else:
            group_lookup[idx] = 2  # consonant
    return group_lookup


def ctc_extended_target(target, blank_id=0):
    extended = [blank_id]
    for label in target:
        extended.append(int(label))
        extended.append(blank_id)
    return torch.tensor(extended, dtype=torch.long)


def forced_align_ctc_viterbi(log_probs, target, blank_id=0):
    ext = ctc_extended_target(target, blank_id=blank_id).to(log_probs.device)

    T = log_probs.shape[0]
    S = ext.shape[0]
    neg_inf = torch.tensor(-1e30, device=log_probs.device, dtype=log_probs.dtype)

    dp = torch.full((T, S), neg_inf, dtype=log_probs.dtype, device=log_probs.device)
    backptr = torch.full((T, S), -1, dtype=torch.long, device=log_probs.device)

    dp[0, 0] = log_probs[0, ext[0]]
    if S > 1:
        dp[0, 1] = log_probs[0, ext[1]]

    for t in range(1, T):
        for s in range(S):
            best_score = dp[t - 1, s]
            best_prev = s

            if s - 1 >= 0 and dp[t - 1, s - 1] > best_score:
                best_score = dp[t - 1, s - 1]
                best_prev = s - 1

            if s - 2 >= 0 and ext[s] != blank_id and ext[s] != ext[s - 2]:
                if dp[t - 1, s - 2] > best_score:
                    best_score = dp[t - 1, s - 2]
                    best_prev = s - 2

            dp[t, s] = best_score + log_probs[t, ext[s]]
            backptr[t, s] = best_prev

    if S == 1:
        end_state = 0
    else:
        end_state = S - 1 if dp[T - 1, S - 1] >= dp[T - 1, S - 2] else S - 2

    state_path = torch.empty(T, dtype=torch.long, device=log_probs.device)
    state_path[-1] = end_state
    for t in range(T - 1, 0, -1):
        state_path[t - 1] = backptr[t, state_path[t]]

    aligned = ext[state_path]
    return aligned.cpu()


def boundary_and_interior_masks(aligned_target, boundary_radius=2, blank_id=0):
    T = aligned_target.numel()
    transition_centers = []

    for t in range(1, T):
        if int(aligned_target[t]) != int(aligned_target[t - 1]):
            transition_centers.append(t)

    boundary = torch.zeros(T, dtype=torch.bool)
    for center in transition_centers:
        start = max(0, center - boundary_radius)
        end = min(T, center + boundary_radius + 1)
        boundary[start:end] = True

    interior = (~boundary) & (aligned_target != blank_id)
    return boundary, interior


def analyze_sample(probabilities, predicted_classes, target_sequence, input_len, target_len, group_lookup, blank_id=0):
    probs = probabilities[:input_len]
    pred = predicted_classes[:input_len].to(torch.long)
    target = target_sequence[:target_len].to(torch.long)

    log_probs = torch.log(torch.clamp(probs, min=1e-8))
    aligned_target = forced_align_ctc_viterbi(log_probs, target, blank_id=blank_id)

    boundary_mask, interior_mask = boundary_and_interior_masks(
        aligned_target,
        boundary_radius=BOUNDARY_RADIUS,
        blank_id=blank_id,
    )

    correctness_mask = pred.eq(aligned_target)
    blank_probability = probs[:, blank_id]

    phoneme_group = torch.tensor(
        [group_lookup[int(x)] for x in aligned_target.tolist()],
        dtype=torch.uint8,
    )

    return {
        "predicted_class": pred.to(torch.int16),
        "blank_probability": blank_probability.to(torch.float32),
        "assigned_target_phoneme": aligned_target.to(torch.int16),
        "boundary_mask": boundary_mask,
        "interior_mask": interior_mask,
        "correctness_mask": correctness_mask,
        "phoneme_group": phoneme_group,
    }


def pad_1d_tensors(tensors, pad_value, dtype=None):
    max_len = max(t.numel() for t in tensors)
    dtype = dtype or tensors[0].dtype
    out = torch.full((len(tensors), max_len), pad_value, dtype=dtype)
    for i, t in enumerate(tensors):
        out[i, : t.numel()] = t.to(dtype)
    return out


payload = load_export(INPUT_PATH)

logits = payload["logits"]
probabilities = payload["probabilities"]
argmax_predictions = payload["argmax_predictions"]
decoded_ctc_sequences = payload["decoded_ctc_sequences"]
target_sequences = payload["target_sequences"]
input_lengths = payload["input_lengths"].to(torch.long)
target_lengths = payload["target_lengths"].to(torch.long)
phoneme_vocab = payload["phoneme_vocab"]

print(f"num_samples: {logits.shape[0]}")
print(f"max_input_frames: {logits.shape[1]}")
print(f"num_classes: {logits.shape[2]}")

group_lookup = build_group_lookup(phoneme_vocab)

per_sample = []
total_samples = logits.shape[0]
total_frames = int(input_lengths.sum().item())
start_time = time.time()
report_every = max(1, total_samples // 20)

pbar = tqdm(range(total_samples), desc="Computing frame metadata", unit="sample")
for i in pbar:
    sample_meta = analyze_sample(
        probabilities=probabilities[i],
        predicted_classes=argmax_predictions[i],
        target_sequence=target_sequences[i],
        input_len=int(input_lengths[i]),
        target_len=int(target_lengths[i]),
        group_lookup=group_lookup,
        blank_id=BLANK_ID,
    )
    per_sample.append(sample_meta)

    if (i + 1) % report_every == 0 or (i + 1) == total_samples:
        elapsed = time.time() - start_time
        samples_done = i + 1
        frames_done = int(input_lengths[:samples_done].sum().item())
        samples_per_sec = samples_done / max(elapsed, 1e-8)
        frames_per_sec = frames_done / max(elapsed, 1e-8)
        eta_sec = (total_samples - samples_done) / max(samples_per_sec, 1e-8)

        pbar.set_postfix({
            "samples/s": f"{samples_per_sec:.2f}",
            "frames/s": f"{frames_per_sec:.0f}",
            "eta_s": f"{eta_sec:.0f}",
        })

analysis_payload = {
    "logits": logits,
    "probabilities": probabilities,
    "argmax_predictions": argmax_predictions,
    "decoded_ctc_sequences": decoded_ctc_sequences,
    "target_sequences": target_sequences,
    "input_lengths": input_lengths,
    "target_lengths": target_lengths,
    "phoneme_vocab": phoneme_vocab,
    "group_label_map": {0: "blank", 1: "vowel", 2: "consonant"},
    "metadata": {
        "predicted_class": pad_1d_tensors([x["predicted_class"] for x in per_sample], pad_value=-1, dtype=torch.int16),
        "blank_probability": pad_1d_tensors([x["blank_probability"] for x in per_sample], pad_value=0.0, dtype=torch.float32),
        "assigned_target_phoneme": pad_1d_tensors([x["assigned_target_phoneme"] for x in per_sample], pad_value=-1, dtype=torch.int16),
        "boundary_mask": pad_1d_tensors([x["boundary_mask"] for x in per_sample], pad_value=False, dtype=torch.bool),
        "interior_mask": pad_1d_tensors([x["interior_mask"] for x in per_sample], pad_value=False, dtype=torch.bool),
        "correctness_mask": pad_1d_tensors([x["correctness_mask"] for x in per_sample], pad_value=False, dtype=torch.bool),
        "phoneme_group": pad_1d_tensors([x["phoneme_group"] for x in per_sample], pad_value=255, dtype=torch.uint8),
    },
}

print("Saving analysis-ready payload...")
save_t0 = time.time()
torch.save(analysis_payload, OUTPUT_PATH)
print(f"Saved in {time.time() - save_t0:.2f}s")

total_elapsed = time.time() - start_time
print(f"total_samples_processed: {total_samples}")
print(f"total_frames_processed: {total_frames}")
print(f"total_runtime_sec: {total_elapsed:.2f}")
print(f"samples_per_sec: {total_samples / max(total_elapsed, 1e-8):.2f}")
print(f"frames_per_sec: {total_frames / max(total_elapsed, 1e-8):.0f}")

print(f"logits: {tuple(analysis_payload['logits'].shape)}")
print(f"probabilities: {tuple(analysis_payload['probabilities'].shape)}")
print(f"argmax_predictions: {tuple(analysis_payload['argmax_predictions'].shape)}")
print(f"decoded_ctc_sequences: {tuple(analysis_payload['decoded_ctc_sequences'].shape)}")
print(f"target_sequences: {tuple(analysis_payload['target_sequences'].shape)}")
print(f"metadata.predicted_class: {tuple(analysis_payload['metadata']['predicted_class'].shape)}")
print(f"metadata.blank_probability: {tuple(analysis_payload['metadata']['blank_probability'].shape)}")
print(f"metadata.assigned_target_phoneme: {tuple(analysis_payload['metadata']['assigned_target_phoneme'].shape)}")
print(f"metadata.boundary_mask: {tuple(analysis_payload['metadata']['boundary_mask'].shape)}")
print(f"metadata.interior_mask: {tuple(analysis_payload['metadata']['interior_mask'].shape)}")
print(f"metadata.correctness_mask: {tuple(analysis_payload['metadata']['correctness_mask'].shape)}")
print(f"metadata.phoneme_group: {tuple(analysis_payload['metadata']['phoneme_group'].shape)}")
print(f"output_file: {OUTPUT_PATH}")


Loading validation export from: /home/alex/snn_for_bts/brain_to_text/model_training/logs/rnn/2026-04-03_17-52-31_037802_rnn_pid114858/checkpoints/validation_only_ctc_export.pt
Loaded export in 0.26s
num_samples: 1426
max_input_frames: 593
num_classes: 41


Computing frame metadata:   0%|          | 0/1426 [00:00<?, ?sample/s]

Saving analysis-ready payload...
Saved in 0.21s
total_samples_processed: 1426
total_frames_processed: 324625
total_runtime_sec: 399.55
samples_per_sec: 3.57
frames_per_sec: 812
logits: (1426, 593, 41)
probabilities: (1426, 593, 41)
argmax_predictions: (1426, 593)
decoded_ctc_sequences: (1426, 67)
target_sequences: (1426, 67)
metadata.predicted_class: (1426, 593)
metadata.blank_probability: (1426, 593)
metadata.assigned_target_phoneme: (1426, 593)
metadata.boundary_mask: (1426, 593)
metadata.interior_mask: (1426, 593)
metadata.correctness_mask: (1426, 593)
metadata.phoneme_group: (1426, 593)
output_file: /home/alex/snn_for_bts/brain_to_text/model_training/logs/rnn/2026-04-03_17-52-31_037802_rnn_pid114858/checkpoints/validation_only_ctc_analysis_ready.pt
